In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib


In [2]:
df = pd.read_csv("../data/gene_expression/biofilm_ml_features.csv")
print("Loaded:", df.shape)
df.head()


Loaded: (5983, 16)


,gene_symbol,biofilm_logFC,biofilm_padj,planktonic_logFC,planktonic_padj,biofilm_abs_logFC,planktonic_abs_logFC,biofilm_significant,planktonic_significant,logFC_interaction,abs_logFC_interaction,logFC_difference,abs_logFC_difference,logFC_ratio,abs_logFC_ratio,biofilm_label
0,PA14_22460,4.345,1.200000e-67,4.345,0.05,4.345,4.345,1,0,18.879025,18.879025,0.0,0.0,1.0,1.0,1
1,ppiA,4.151,3.000000e-63,4.151,0.05,4.151,4.151,1,0,17.230801,17.230801,0.0,0.0,1.0,1.0,1
2,PA14_22470,3.929,1.700000e-62,3.929,0.05,3.929,3.929,1,0,15.437041,15.437041,0.0,0.0,1.0,1.0,1
3,bapA,3.773,4.400000e-59,3.773,0.05,3.773,3.773,1,0,14.235529,14.235529,0.0,0.0,1.0,1.0,1
4,PA14_27070,3.263,1.600000e-46,3.263,0.05,3.263,3.263,1,0,10.647169,10.647169,0.0,0.0,1.0,1.0,1


In [3]:
# ------------------------------------------------------------
# Step 2: Define X (features) and y (target)
# ------------------------------------------------------------

feature_cols = [
    "biofilm_logFC", "biofilm_padj",
    "planktonic_logFC", "planktonic_padj",
    "biofilm_abs_logFC", "planktonic_abs_logFC",
    "biofilm_significant", "planktonic_significant",
    "logFC_interaction", "abs_logFC_interaction",
    "logFC_difference", "abs_logFC_difference",
    "logFC_ratio", "abs_logFC_ratio"
]

X = df[feature_cols]
y = df["biofilm_label"]

print("Features:", X.shape)
print("Target:", y.shape)


Features: (5983, 14)
Target: (5983,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


Train: (4786, 14)
Test: (1197, 14)


In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [6]:
log_reg = LogisticRegression(max_iter=500)
log_reg.fit(X_train_scaled, y_train)

log_pred = log_reg.predict(X_test_scaled)
log_acc = accuracy_score(y_test, log_pred)

print("Logistic Regression Accuracy:", log_acc)
print(classification_report(y_test, log_pred))


Logistic Regression Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1193
           1       1.00      1.00      1.00         4

    accuracy                           1.00      1197
   macro avg       1.00      1.00      1.00      1197
weighted avg       1.00      1.00      1.00      1197



In [7]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_acc)
print(classification_report(y_test, rf_pred))


Random Forest Accuracy: 0.9991645781119465
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1193
           1       1.00      0.75      0.86         4

    accuracy                           1.00      1197
   macro avg       1.00      0.88      0.93      1197
weighted avg       1.00      1.00      1.00      1197



In [8]:
if rf_acc >= log_acc:
    best_model = rf
    best_name = "random_forest.pkl"
else:
    best_model = log_reg
    best_name = "logistic_regression.pkl"

print("Best model:", best_name)


Best model: logistic_regression.pkl


In [9]:
joblib.dump(best_model, f"../models/{best_name}")
print("Saved model:", best_name)


Saved model: logistic_regression.pkl


In [13]:
import joblib
model = joblib.load("../models/logistic_regression.pkl")
